# Image Storage Processing 

In [1]:
# import libraries for data processing
import numpy as np
import pandas as pd

# import libraries for file manipulation + system management
import os
import requests
import shutil
from urllib.parse import urlparse

# import dependency from image processing toolkit
from image_processing_tools import distribute_images
from image_processing_tools import remove_images
from image_processing_tools import det_folder_size

### ISP-Approach

+ Loading Images
+ Download images into source images (IF NEEDED)
+ Distributing stored images from source path to new destination path
+ Image Reorganisation based on labels

## Loading Images 

In [2]:
# load dataset 
image_ds_file = "image_dataset.csv"
image_df = pd.read_csv(image_ds_file, index_col=0)
image_df.head()

,Image_file,Labels,Model,Company
0,image_000.jpg,Samsung SM-A156B Galaxy A15 Dual SIM 5G 4GB RA...,SM-A156B Galaxy,Samsung
1,image_001.jpg,Samsung SM-A556B Galaxy A55 5G Dual SIM 8GB 12...,SM-A556B Galaxy,Samsung
2,image_002.jpg,Motorola Moto G54 256GB Blue 5G Android Smartp...,Moto G54,Motorola
3,image_003.jpg,Xiaomi 14 Ultra 5G 16GB/512GB White (White) Du...,14 Ultra,Xiaomi
4,image_004.jpg,Samsung Galaxy S20 FE 5G 6GB/128GB Purple (Lav...,Galaxy S20,Samsung


In [3]:
image_df["Company"].value_counts()

Company
Samsung      345
Xiaomi       167
Apple        105
Nokia        103
Motorola      30
OnePlus       26
Redmi         13
Poco          10
Microsoft      1
Name: count, dtype: int64

## Distribution of stored images


In [4]:
# data splitting: create constants for number of images in train, test and validation folders 
train_size = 800
test_size = 160
validation_size = 40    

In [5]:
# create list of folders
path = "D:\\Machine_Learning\\Portfolio_Project_Machine_Learning\\Mobile_Phone_Recognition\\datasets"
dataset_paths = os.listdir(path)
dataset_paths.remove("dataset_structure")
print(f"Dataset Paths: {dataset_paths}")

# create constants: source folder and destination folder (train, test and validation folder) 
source_path = os.path.join(path, dataset_paths[1])
destination_path = os.path.join(path, dataset_paths[0])
  
for folder_name, path_idx in zip(["Destination Folder", "Source Folder"], dataset_paths):
  ds_path = os.path.join(path, path_idx)
  print(f"{folder_name} Folder: {ds_path}")

Dataset Paths: ['distributed_images', 'source_images', 'source_images_v1', 'source_images_v2']
Destination Folder Folder: D:\Machine_Learning\Portfolio_Project_Machine_Learning\Mobile_Phone_Recognition\datasets\distributed_images
Source Folder Folder: D:\Machine_Learning\Portfolio_Project_Machine_Learning\Mobile_Phone_Recognition\datasets\source_images


In [6]:
# update 1: train and test sizes
train_size1 = 0.8
test_size1 = 0.2

# update 2: train and test images
train_size2 = 0.7
test_size2 = 0.3

In [7]:
# remove images from current folder 
remove_images(current_folder=destination_path)

In [8]:
# distribute images in training, testing and validation folder  
distribute_images(source_folder=source_path, 
                  destination_folder=destination_path, 
                  df_file=image_ds_file,
                  update_train_size=train_size2,
                  update_test_size=test_size2)

Moved image_000.jpg to train\Samsung
Moved image_000.jpg to train\Samsung
Moved image_001.jpg to train\Samsung
Moved image_003.jpg to train\Xiaomi
Moved image_002.jpg to train\Motorola
Moved image_001.jpg to train\Samsung
Moved image_004.jpg to train\Samsung
Moved image_004.jpg to train\Samsung
Moved image_006.jpg to train\Samsung
Moved image_007.jpg to train\Samsung
Moved image_005.jpg to train\Samsung
Moved image_003.jpg to train\Xiaomi
Moved image_006.jpg to train\Samsung
Moved image_000.jpg to train\Samsung
Moved image_009.jpg to train\Samsung
Moved image_002.jpg to train\Motorola
Moved image_014.jpg to train\Samsung
Moved image_013.jpg to train\Xiaomi
Moved image_008.jpg to train\Samsung
Moved image_016.jpg to train\Samsung
Moved image_007.jpg to train\Samsung
Moved image_010.jpg to train\Samsung
Moved image_009.jpg to train\Samsung
Moved image_021.jpg to train\Xiaomi
Moved image_015.jpg to train\Samsung
Moved image_012.jpg to train\Xiaomi
Moved image_005.jpg to train\Samsung
Move

Redistributing new images with updated train and test sizes:
+ Remove the remain exissting images with the old train and test sizes 
+ Update the train and test sizes
+ Redistribute the images with updated train and test sizes

## Reorganising images based on its label

+ finding number of images per training, testing and validation folders
+ finding number of images per class label folders

In [9]:
distributed_path = "D:\\Machine_Learning\\Portfolio_Project_Machine_Learning\\Mobile_Phone_Recognition\\datasets\\distributed_images"
distributed_path_list = os.listdir(distributed_path)

train_path = os.path.join(distributed_path, "train")
test_path = os.path.join(distributed_path, "test")
validation_pth = os.path.join(distributed_path, "validation")

print(f"Training folder: {train_path}")
print(f"Testing folder: {test_path}")
# print(f"Validation folder: {validation_path}")

Training folder: D:\Machine_Learning\Portfolio_Project_Machine_Learning\Mobile_Phone_Recognition\datasets\distributed_images\train
Testing folder: D:\Machine_Learning\Portfolio_Project_Machine_Learning\Mobile_Phone_Recognition\datasets\distributed_images\test


In [15]:
# define folder size: train, test, validation 
image_train_counter = det_folder_size(sel_path=train_path)
image_test_counter = det_folder_size(sel_path=test_path)
# image_val_counter = det_folder_size(sel_path=validation_pth)

# convert into dataframe
image_counter_df = pd.DataFrame()
image_counter_df["Image_Labels"] = image_train_counter.keys()
image_counter_df["Train_size"] = image_train_counter.values()
image_counter_df["Test_size"] = image_test_counter.values()
# image_counter_df["Validation_size"] = image_val_counter.values() 

In [16]:
image_counter_df

,Image_Labels,Train_size,Test_size
0,Apple,24,26
1,Motorola,14,8
2,Nokia,26,27
3,OnePlus,7,11
4,Poco,3,3
5,Redmi,7,2
6,Samsung,144,86
7,Xiaomi,51,28


In [17]:
image_counter_df["Train_size"].sum()

np.int64(276)

In [18]:
image_total_set = pd.DataFrame()
image_total_set["Train_sum"] = pd.Series(image_counter_df["Train_size"].sum())
image_total_set["Test_sum"] = pd.Series(image_counter_df["Test_size"].sum())
# image_total_set["Valdation_sum"] = pd.Series(image_counter_df["Validation_size"].sum())

# image_total_set = image_total_set.transpose()
image_total_set.transpose()

,0
Train_sum,276
Test_sum,191
